# Mouse Protein Expression: Data Audit

This notebook explores the UCI Mice Protein Expression dataset before machine-learning modelling.

## Objectives

- Inspect the dataset structure and class labels.
- identify missing protein-expression measurements.
- examine repeated measurements from the same mouse.
- prevent data leakage during model evaluation.
# Leakage-Aware Classification of Mouse Protein Expression

## Project overview

This project develops a machine-learning workflow for classifying experimental mouse groups using cerebral-cortex protein-expression measurements.

The main emphasis is not only model performance, but also biologically appropriate evaluation. Because each mouse contributes multiple measurements, all measurements from the same mouse must remain together during model training and testing. This prevents subject-level data leakage and provides a more realistic estimate of performance on unseen biological mice.

The dataset is obtained from the [UCI Machine Learning Repository: Mice Protein Expression](https://archive.ics.uci.edu/dataset/342/mice+protein+expression).

---

## Biological question

Can the combined expression profiles of 77 cerebral-cortex proteins distinguish eight experimental mouse groups defined by genotype, learning paradigm, and treatment?

A traditional analysis might examine individual proteins separately. Machine learning allows the combined expression pattern across all 77 proteins to be evaluated simultaneously.

---

## Dataset structure

The dataset contains:

- 72 biological mice.
- 15 repeated measurements per mouse.
- 1,080 total measurement rows.
- 77 protein-expression variables.
- 8 experimental target classes.

The original UCI feature table also includes three experimental descriptors:

- `Genotype`
- `Treatment`
- `Behavior`

These descriptors are not used as model predictors because they directly define the target class. Including them would allow the model to access the answer and cause target leakage.

---

## Experimental class labels

Each target label combines three experimental properties:

| Label component | Meaning |
|---|---|
| `c` | Control mouse |
| `t` | Trisomic mouse |
| `CS` | Context-shock learning paradigm |
| `SC` | Shock-context learning paradigm |
| `m` | Memantine treatment |
| `s` | Saline treatment |

For example, `t-CS-m` represents a trisomic mouse subjected to the context-shock paradigm and treated with memantine.

The complete set of target classes is:

- `c-CS-m`
- `c-CS-s`
- `c-SC-m`
- `c-SC-s`
- `t-CS-m`
- `t-CS-s`
- `t-SC-m`
- `t-SC-s`

---

## Machine-learning input and output

### Model input

The model receives the 77 protein-expression measurements.

It does not receive:

- Mouse identifiers
- Genotype
- Treatment
- Behavior
- Experimental class

Mouse identifiers are used only to organize the training and testing groups.

### Model output

For an unseen measurement, the model produces:

1. A predicted experimental class.
2. An estimated probability for each of the eight classes.

Because each mouse contributes 15 measurements, predictions can also be combined to produce one final mouse-level prediction. Class probabilities from the measurements of the same mouse can be averaged, and the class with the highest average probability can be selected.

---

## Why subject-level grouping is necessary

The 1,080 rows are not 1,080 independent biological samples. They originate from only 72 mice.

Measurements from the same mouse are likely to be more similar to each other than measurements from different mice. A random row-level split could place some measurements from one mouse in the training set and other measurements from that same mouse in the test set.

In that situation, the model would be evaluated partly on mice it had already encountered. This could produce an overly optimistic performance estimate.

The primary evaluation therefore follows this rule:

> All measurements from one biological mouse must belong exclusively to either the training set or the test set.

A random row-level split may be examined only as a demonstration of how subject-level leakage can inflate apparent performance. It will not be treated as the valid final evaluation.

---

## Missing-value strategy

Some protein-expression measurements are missing.

Missing values will not be replaced before separating training and testing mice. Instead, imputation will be included inside the machine-learning pipeline:

1. Separate mice into training and testing groups.
2. Calculate imputation values using the training data only.
3. Apply those training-derived values to both training and testing data.

This prevents information from the test mice from influencing model development.

---

## Planned analytical workflow

1. Load the dataset and inspect its metadata.
2. Separate identifiers, predictors, target labels, and grouping variables.
3. Retain only the 77 protein-expression predictors.
4. Identify repeated measurements belonging to the same mouse.
5. Audit missing values and class balance.
6. Split the data by biological mouse.
7. Fit preprocessing operations using training data only.
8. Train a simple interpretable baseline model.
9. Compare additional classification models where appropriate.
10. Evaluate measurement-level and mouse-level performance.
11. Examine confusion matrices and class-specific performance.
12. Investigate which proteins contribute to classification.
13. Document limitations and avoid causal interpretation.

---

## Planned evaluation

Model performance will be assessed using measures suitable for multiclass classification, including:

- Balanced accuracy
- Macro-averaged F1 score
- Confusion matrix
- Class-specific precision and recall

Where possible, performance will be reported at both:

- The individual-measurement level
- The biological-mouse level

The mouse-level evaluation will be treated as the more biologically meaningful result.

---

## Scientific interpretation

The analysis tests whether protein-expression patterns contain enough information to distinguish the experimental groups.

It does not establish that proteins identified as important:

- Cause the experimental differences.
- Represent validated biomarkers.
- Provide a clinical or diagnostic test.

Feature importance in a predictive model indicates an association with classification, not a causal biological mechanism.

---

## Project limitations

- The dataset contains only 72 independent biological mice.
- The eight experimental classes are already known.
- The project is therefore a proof-of-concept molecular-pattern analysis rather than a clinical diagnostic application.
- Repeated measurements increase the number of rows but do not increase the number of independent biological subjects.
- Model results require cautious interpretation and independent experimental validation.

---

## Project objective

This portfolio project demonstrates a reproducible and biologically informed approach to machine learning, with particular emphasis on:

- Biological experimental units
- Repeated measurements
- Missing-data handling
- Target leakage
- Group-aware evaluation
- Transparent model interpretation

In [74]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

mice = fetch_ucirepo(id=342)

X = mice.data.features
y = mice.data.targets
ids = mice.data.ids

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [75]:
display(
    mice.variables[
        ["name", "role", "type", "missing_values"]
    ].head(10)
)
display(mice.variables[["name", "role"]].tail(10))

,name,role,type,missing_values
0,MouseID,ID,Categorical,no
1,DYRK1A_N,Feature,Continuous,yes
2,ITSN1_N,Feature,Continuous,no
3,BDNF_N,Feature,Continuous,yes
4,NR1_N,Feature,Continuous,no
5,NR2A_N,Feature,Continuous,yes
6,pAKT_N,Feature,Continuous,no
7,pBRAF_N,Feature,Continuous,yes
8,pCAMKII_N,Feature,Continuous,no
9,pCREB_N,Feature,Continuous,yes


,name,role
72,pCFOS_N,Feature
73,SYP_N,Feature
74,H3AcK18_N,Feature
75,EGR1_N,Feature
76,H3MeK4_N,Feature
77,CaNA_N,Feature
78,Genotype,Feature
79,Treatment,Feature
80,Behavior,Feature
81,class,Target


In [76]:
print("Object type:")
print(type(mice))

print("\nMain sections:")
print(mice.keys())

print("\nSections inside mice.data:")
print(mice.data.keys())

Object type:
<class 'ucimlrepo.dotdict.dotdict'>

Main sections:
dict_keys(['data', 'metadata', 'variables'])

Sections inside mice.data:
dict_keys(['ids', 'features', 'targets', 'original', 'headers'])


## 1. Dataset structure

In [77]:
print("Feature-table shape:", X.shape)
print("Target-table shape:", y.shape)

print("\nID columns:")
print(ids.columns.tolist())

print("\nTarget columns:")
print(y.columns.tolist())

display(ids.head())
display(X.head())
display(y.head())

Feature-table shape: (1080, 80)
Target-table shape: (1080, 1)

ID columns:
['MouseID']

Target columns:
['class']


,MouseID
0,309_1
1,309_2
2,309_3
3,309_4
4,309_5


,DYRK1A_N,ITSN1_N,BDNF_N,NR1_N,NR2A_N,pAKT_N,pBRAF_N,pCAMKII_N,pCREB_N,pELK_N,...,pS6_N,pCFOS_N,SYP_N,H3AcK18_N,EGR1_N,H3MeK4_N,CaNA_N,Genotype,Treatment,Behavior
0,0.503644,0.747193,0.430175,2.816329,5.990152,0.218830,0.177565,2.373744,0.232224,1.750936,...,0.106305,0.108336,0.427099,0.114783,0.131790,0.128186,1.675652,Control,Memantine,C/S
1,0.514617,0.689064,0.411770,2.789514,5.685038,0.211636,0.172817,2.292150,0.226972,1.596377,...,0.106592,0.104315,0.441581,0.111974,0.135103,0.131119,1.743610,Control,Memantine,C/S
2,0.509183,0.730247,0.418309,2.687201,5.622059,0.209011,0.175722,2.283337,0.230247,1.561316,...,0.108303,0.106219,0.435777,0.111883,0.133362,0.127431,1.926427,Control,Memantine,C/S
3,0.442107,0.617076,0.358626,2.466947,4.979503,0.222886,0.176463,2.152301,0.207004,1.595086,...,0.103184,0.111262,0.391691,0.130405,0.147444,0.146901,1.700563,Control,Memantine,C/S
4,0.434940,0.617430,0.358802,2.365785,4.718679,0.213106,0.173627,2.134014,0.192158,1.504230,...,0.104784,0.110694,0.434154,0.118481,0.140314,0.148380,1.839730,Control,Memantine,C/S


,class
0,c-CS-m
1,c-CS-m
2,c-CS-m
3,c-CS-m
4,c-CS-m


## 2. Select the protein-expression predictors

The UCI feature table includes genotype, treatment, and behavior alongside the protein-expression measurements. Because the class label is constructed from these three experimental descriptors, using them to predict the class would cause target leakage. Therefore, only the 77 protein-expression variables are retained as model predictors.

In [78]:
metadata_columns = ["Genotype", "Treatment", "Behavior"]

X_proteins = X.drop(columns=metadata_columns)

print("Original feature-table shape:", X.shape)
print("Protein-only feature-table shape:", X_proteins.shape)

Original feature-table shape: (1080, 80)
Protein-only feature-table shape: (1080, 77)


## 3. Define the target and biological grouping variable

The modelling target is the eight-class experimental label. Because each biological mouse contributes 15 measurements, the numerical mouse identifier is extracted from `MouseID`. This grouping variable will later prevent measurements from the same mouse from appearing in both training and testing sets.

In [79]:
y_class = y["class"]

mouse_groups = ids["MouseID"].str.split("_").str[0]

print("Number of target classes:", y_class.nunique())
print("Class labels:", sorted(y_class.unique()))

print("\nUnique measurement IDs:", ids["MouseID"].nunique())
print("Unique biological mice:", mouse_groups.nunique())

Number of target classes: 8
Class labels: ['c-CS-m', 'c-CS-s', 'c-SC-m', 'c-SC-s', 't-CS-m', 't-CS-s', 't-SC-m', 't-SC-s']

Unique measurement IDs: 1080
Unique biological mice: 72


In [80]:
measurements_per_mouse = mouse_groups.value_counts()

print("Minimum measurements per mouse:", measurements_per_mouse.min())
print("Maximum measurements per mouse:", measurements_per_mouse.max())
print("Do all mice have 15 measurements?", measurements_per_mouse.eq(15).all())

Minimum measurements per mouse: 15
Maximum measurements per mouse: 15
Do all mice have 15 measurements? True


## 4. Audit missing protein measurements

Missing measurements must be identified before modelling. Any imputation will later be fitted using only the training data to prevent information from the test set leaking into the model.

In [81]:
missing_per_protein = (X_proteins.isna().sum().sort_values(ascending=False))

proteins_with_missing = missing_per_protein[missing_per_protein > 0]

print("Total missing values:", missing_per_protein.sum())
print("Proteins with missing values:", proteins_with_missing.shape[0])

display(proteins_with_missing.head(10).to_frame(name="Missing measurements"))

Total missing values: 1396
Proteins with missing values: 49


,Missing measurements
BCL2_N,285
H3MeK4_N,270
BAD_N,213
EGR1_N,210
H3AcK18_N,180
pCFOS_N,75
ELK_N,18
Bcatenin_N,18
MEK_N,7
ITSN1_N,3


## 5. Examine class balance

Class balance is checked at both the measurement level and the biological-mouse level. This is important because every mouse contributes multiple measurements.

In [82]:
class_measurement_counts = y_class.value_counts()

display(class_measurement_counts.to_frame(name="Number of measurements"))

,Number of measurements
class,
c-CS-m,150
c-SC-m,150
c-CS-s,135
c-SC-s,135
t-CS-m,135
t-SC-m,135
t-SC-s,135
t-CS-s,105


In [83]:
mouse_class_pairs = pd.DataFrame({
    "Mouse": mouse_groups,
    "Class": y_class
})

unique_mouse_class_pairs = mouse_class_pairs.drop_duplicates()

print("Rows before removing repetitions:", mouse_class_pairs.shape[0])
print("Rows after removing repetitions:", unique_mouse_class_pairs.shape[0])


Rows before removing repetitions: 1080
Rows after removing repetitions: 72


In [84]:
class_mouse_counts = (
    unique_mouse_class_pairs["Class"]
    .value_counts()
    .sort_index()
)

display(
    class_mouse_counts.to_frame(
        name="Number of biological mice"
    )
)

,Number of biological mice
Class,
c-CS-m,10
c-CS-s,9
c-SC-m,10
c-SC-s,9
t-CS-m,9
t-CS-s,7
t-SC-m,9
t-SC-s,9


In [85]:
class_balance = pd.DataFrame({
    "Measurements": class_measurement_counts,
    "Biological mice": class_mouse_counts
}).sort_index()

display(class_balance)

,Measurements,Biological mice
c-CS-m,150,10
c-CS-s,135,9
c-SC-m,150,10
c-SC-s,135,9
t-CS-m,135,9
t-CS-s,105,7
t-SC-m,135,9
t-SC-s,135,9


## 6. Inspect protein-expression distributions

Summary statistics are examined to understand the ranges and variability of the protein measurements. This can reveal different measurement scales, unusually large ranges, and potential outliers. Unusual values will not be removed automatically because they may represent genuine biological variation.

In [86]:
protein_summary = X_proteins.describe().T

display(protein_summary.head(10))
display(protein_summary.tail(10))

,count,mean,std,min,25%,50%,75%,max
DYRK1A_N,1077.0,0.425810,0.249362,0.145327,0.288121,0.366378,0.487711,2.516367
ITSN1_N,1077.0,0.617102,0.251640,0.245359,0.473361,0.565782,0.698032,2.602662
BDNF_N,1077.0,0.319088,0.049383,0.115181,0.287444,0.316564,0.348197,0.497160
NR1_N,1077.0,2.297269,0.347293,1.330831,2.057411,2.296546,2.528481,3.757641
NR2A_N,1077.0,3.843934,0.933100,1.737540,3.155678,3.760855,4.440011,8.482553
pAKT_N,1077.0,0.233168,0.041634,0.063236,0.205755,0.231177,0.257261,0.539050
pBRAF_N,1077.0,0.181846,0.027042,0.064043,0.164595,0.182302,0.197418,0.317066
pCAMKII_N,1077.0,3.537109,1.295169,1.343998,2.479834,3.326520,4.481940,7.464070
pCREB_N,1077.0,0.212574,0.032587,0.112812,0.190823,0.210594,0.234595,0.306247
pELK_N,1077.0,1.428682,0.466904,0.429032,1.203665,1.355846,1.561316,6.113347


,count,mean,std,min,25%,50%,75%,max
SHH_N,1080.0,0.226676,0.028989,0.155869,0.206395,0.224000,0.241655,0.358289
BAD_N,867.0,0.157914,0.029537,0.088305,0.136424,0.152313,0.174017,0.282016
BCL2_N,795.0,0.134762,0.027417,0.080657,0.115554,0.129468,0.148235,0.261506
pS6_N,1080.0,0.121521,0.014276,0.067254,0.110839,0.121626,0.131955,0.158748
pCFOS_N,1005.0,0.131053,0.023863,0.085419,0.113506,0.126523,0.143652,0.256529
SYP_N,1080.0,0.446073,0.066432,0.258626,0.398082,0.448459,0.490773,0.759588
H3AcK18_N,900.0,0.169609,0.059402,0.079691,0.125848,0.158240,0.197876,0.479763
EGR1_N,870.0,0.183135,0.040406,0.105537,0.155121,0.174935,0.204542,0.360692
H3MeK4_N,810.0,0.205440,0.055514,0.101787,0.165143,0.193994,0.235215,0.413903
CaNA_N,1080.0,1.337784,0.317126,0.586479,1.081423,1.317441,1.585824,2.129791


## 7. Check for non-informative protein variables

Protein variables with only one observed value contain no variation and therefore cannot contribute to classification.

In [87]:
unique_values_per_protein = X_proteins.nunique()

constant_proteins = unique_values_per_protein[
    unique_values_per_protein <= 1
]

print("Number of constant proteins:", constant_proteins.shape[0])

display(
    constant_proteins.to_frame(
        name="Number of unique values"
    )
)


Number of constant proteins: 0


,Number of unique values


### Constant-feature finding

No constant protein variables were detected. All 77 protein predictors contain variation and were retained for subsequent analysis.

## 8. Data-audit conclusions

The audit identified the following properties:

- The dataset contains 1,080 measurements from 72 biological mice.
- Each mouse contributes 15 repeated measurements.
- The modelling input contains 77 protein-expression variables.
- The target contains eight experimental classes.
- Class sizes vary from 7 to 10 biological mice, corresponding to 105–150 repeated measurements per class.
- The dataset therefore has mild class imbalance, which should be considered during model evaluation.
- Missing protein measurements are present and will require imputation.
- Imputation must be learned from the training data only.
- Protein variables have different numerical ranges, so scaling may be required for scale-sensitive models.
- No constant protein variables were detected.
- All measurements from one mouse must remain in the same data partition to prevent subject-level leakage.
